# Gold — Arquitectura Medallón con Datos de Fraude

**Semana:** 02  
**Actividad:** 04  
**Capa:** Gold  
**Notebook:** gold_daniel  

## Objetivo

Leer la tabla maestra de Silver y construir tablas agregadas orientadas a análisis de negocio sobre fraude.

## Entrada

- `workspace.silver.transactions_daniel`

## Salidas Gold

- `workspace.gold.fraude_por_categoria_daniel`
- `workspace.gold.fraude_por_tarjeta_daniel`
- `workspace.gold.fraude_temporal_daniel`
- `workspace.gold.usuarios_riesgo_daniel`

## Regla Gold

La capa Gold no debe leer archivos fuente ni tablas Bronze.  
Debe leer desde Silver y generar tablas resumidas para consumo analítico.

In [0]:
from pyspark.sql import functions as F

MI_NOMBRE = "daniel"
CATALOG = "workspace"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

df_silver = spark.table(f"{CATALOG}.silver.transactions_{MI_NOMBRE}")

print(f"Filas Silver: {df_silver.count():,}")
print(f"Columnas Silver: {len(df_silver.columns)}")

display(df_silver.limit(5))

In [0]:
df_gold_categoria = (
    df_silver
    .groupBy("merchant_category", "mcc")
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == 1, 1).otherwise(0)).alias("total_fraudes"),
        F.sum(F.when(F.col("is_fraud") == 0, 1).otherwise(0)).alias("total_legitimas"),
        F.sum(F.when(F.col("is_fraud").isNull(), 1).otherwise(0)).alias("sin_label"),
        F.sum(F.when(F.col("is_fraud").isin(0, 1), 1).otherwise(0)).alias("transacciones_etiquetadas"),
        F.round(F.sum("amount"), 2).alias("monto_total"),
        F.round(F.avg("amount"), 2).alias("ticket_promedio")
    )
    .withColumn(
        "tasa_fraude_pct",
        F.when(
            F.col("transacciones_etiquetadas") > 0,
            F.round(F.col("total_fraudes") / F.col("transacciones_etiquetadas") * 100, 4)
        ).otherwise(None)
    )
    .orderBy(F.col("tasa_fraude_pct").desc())
)

df_gold_categoria.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.fraude_por_categoria_{MI_NOMBRE}")

display(df_gold_categoria.limit(10))

In [0]:

   df_gold_tarjeta = (
    df_silver
    .groupBy("card_type")
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == 1, 1).otherwise(0)).alias("total_fraudes"),
        F.sum(F.when(F.col("is_fraud") == 0, 1).otherwise(0)).alias("total_legitimas"),
        F.sum(F.when(F.col("is_fraud").isNull(), 1).otherwise(0)).alias("sin_label"),
        F.sum(F.when(F.col("is_fraud").isin(0, 1), 1).otherwise(0)).alias("transacciones_etiquetadas"),
        F.round(F.avg("amount"), 2).alias("monto_promedio"),
        F.round(F.avg(F.when(F.col("is_fraud") == 1, F.col("amount"))), 2).alias("monto_promedio_fraude")
    )
    .withColumn(
        "tasa_fraude_pct",
        F.when(
            F.col("transacciones_etiquetadas") > 0,
            F.round(F.col("total_fraudes") / F.col("transacciones_etiquetadas") * 100, 4)
        ).otherwise(None)
    )
    .orderBy(F.col("tasa_fraude_pct").desc())
)

df_gold_tarjeta.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.fraude_por_tarjeta_{MI_NOMBRE}")

display(df_gold_tarjeta)

In [0]:
df_gold_temporal = (
    df_silver
    .groupBy("anio", "mes", "dia_semana", "hora", "es_fin_de_semana")
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == 1, 1).otherwise(0)).alias("total_fraudes"),
        F.sum(F.when(F.col("is_fraud") == 0, 1).otherwise(0)).alias("total_legitimas"),
        F.sum(F.when(F.col("is_fraud").isNull(), 1).otherwise(0)).alias("sin_label"),
        F.sum(F.when(F.col("is_fraud").isin(0, 1), 1).otherwise(0)).alias("transacciones_etiquetadas")
    )
    .withColumn(
        "tasa_fraude_pct",
        F.when(
            F.col("transacciones_etiquetadas") > 0,
            F.round(F.col("total_fraudes") / F.col("transacciones_etiquetadas") * 100, 4)
        ).otherwise(None)
    )
    .orderBy("anio", "mes", "dia_semana", "hora")
)

df_gold_temporal.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.fraude_temporal_{MI_NOMBRE}")

display(df_gold_temporal.limit(20))

In [0]:
df_gold_usuarios = (
    df_silver
    .groupBy("user_id")
    .agg(
        F.count("transaction_id").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == 1, 1).otherwise(0)).alias("total_fraudes"),
        F.sum(F.when(F.col("is_fraud") == 0, 1).otherwise(0)).alias("total_legitimas"),
        F.sum(F.when(F.col("is_fraud").isNull(), 1).otherwise(0)).alias("sin_label"),
        F.sum(F.when(F.col("is_fraud").isin(0, 1), 1).otherwise(0)).alias("transacciones_etiquetadas"),
        F.round(F.sum("amount"), 2).alias("monto_total"),
        F.countDistinct("card_id").alias("num_tarjetas")
    )
    .withColumn(
        "tasa_fraude_pct",
        F.when(
            F.col("transacciones_etiquetadas") > 0,
            F.round(F.col("total_fraudes") / F.col("transacciones_etiquetadas") * 100, 4)
        ).otherwise(None)
    )
    .filter(F.col("total_transacciones") >= 5)
    .orderBy(F.col("tasa_fraude_pct").desc(), F.col("total_fraudes").desc())
)

df_gold_usuarios.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.gold.usuarios_riesgo_{MI_NOMBRE}")

display(df_gold_usuarios.limit(10))

In [0]:
print("Tablas Gold disponibles:")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.gold"))

In [0]:
# Validación SQL sobre tablas Gold

# 1. Top 5 categorías de comercio con mayor tasa de fraude
print("Top 5 categorías de comercio con mayor tasa de fraude")
spark.sql(f"""
    SELECT 
        merchant_category, 
        mcc,
        tasa_fraude_pct, 
        total_transacciones, 
        total_fraudes,
        transacciones_etiquetadas
    FROM workspace.gold.fraude_por_categoria_daniel
    WHERE transacciones_etiquetadas > 0
    ORDER BY tasa_fraude_pct DESC
    LIMIT 5
""").show(truncate=False)

# 2. ¿A qué hora del día hay más fraude?
print("Horas con más fraude")
spark.sql(f"""
    SELECT 
        hora, 
        SUM(total_fraudes) AS fraudes_totales,
        SUM(transacciones_etiquetadas) AS transacciones_etiquetadas,
        ROUND(SUM(total_fraudes) / SUM(transacciones_etiquetadas) * 100, 4) AS tasa_fraude_pct
    FROM workspace.gold.fraude_temporal_daniel
    GROUP BY hora
    ORDER BY fraudes_totales DESC
    LIMIT 10
""").show()

# 3. ¿El fraude es más frecuente en fin de semana?
print("Fraude en fin de semana vs entre semana")
spark.sql(f"""
    SELECT 
        es_fin_de_semana,
        SUM(total_fraudes) AS fraudes,
        SUM(transacciones_etiquetadas) AS transacciones_etiquetadas,
        ROUND(SUM(total_fraudes) / SUM(transacciones_etiquetadas) * 100, 4) AS tasa_fraude_pct
    FROM workspace.gold.fraude_temporal_daniel
    GROUP BY es_fin_de_semana
    ORDER BY tasa_fraude_pct DESC
""").show()

## Resultados principales de validación

La categoría de comercio con mayor tasa de fraude fue **Cruise Lines**, con MCC **4411** y una tasa de fraude aproximada de **59.7826%** sobre transacciones etiquetadas.

La hora con mayor cantidad de fraude fue la hora **11**, con **1,610 fraudes**.

El fraude fue relativamente más frecuente en **fin de semana**, con una tasa de **0.1602%**, frente a **0.1453%** entre semana.

Para el cálculo de tasas se usaron únicamente transacciones etiquetadas, evitando que los registros sin etiqueta afecten el denominador.